# Monte Carlo Counterfactual Regret Minimization (MCCFR)

## Algorithm 4: External Sampling with Stochastically-Weighted Averaging

This notebook implements the MCCFR algorithm for poker AI using external sampling. Each code cell corresponds directly to the pseudocode.

### Pseudocode Overview:
1. **Initialize**: `∀I ∈ Z, ∀a ∈ A(I): r_I[a] ← s_I[a] ← 0`
2. **ExternalSampling(h, i)**: Recursive CFR traversal
   - Terminal states: return utility
   - Chance nodes: sample action
   - Traversing player's nodes: compute regrets, update regret table
   - Opponent's nodes: sample from strategy, update strategy table

### Game Tree Structure:
- **h**: History (game state sequence)
- **I**: Information set (what player knows at decision point)  
- **A(I)**: Legal actions at information set I
- **P(h)**: Player to act at history h (or 'c' for chance)
- **σ(I)**: Strategy (probability distribution over actions)
- **r_I[a]**: Cumulative regret for action a at infoset I
- **s_I[a]**: Cumulative strategy for action a at infoset I

In [3]:
# ============================================================================
# IMPORTS AND SETUP
# ============================================================================
import sys
import os
import random
import numpy as np
import time 
import pickle
from collections import defaultdict
from typing import Dict, List, Tuple, Union
from pathlib import Path
import pkrbot

# Add parent directory to path to import custom_engine
sys.path.insert(0, '/Users/nikhileshbelulkar/Documents/mit-poker-2026')

from custom_engine import (
    RoundState, TerminalState, 
    FoldAction, CallAction, CheckAction, RaiseAction, DiscardAction,
    STARTING_STACK, BIG_BLIND, SMALL_BLIND
)

# Import the canonicalization class
sys.path.insert(0, '/Users/nikhileshbelulkar/Documents/mit-poker-2026/deep_CFR_vNB')
from canocalization_board_cards import canon_cards

print("✓ Imports successful!")
print(f"  Using custom_engine with action_taken field")
print(f"  Using canon_cards class for card abstraction")
print(f"  Starting stack: {STARTING_STACK}")
print(f"  Big blind: {BIG_BLIND}")
print(f"  Small blind: {SMALL_BLIND}")

✓ Imports successful!
  Using custom_engine with action_taken field
  Using canon_cards class for card abstraction
  Starting stack: 400
  Big blind: 2
  Small blind: 1


In [4]:
# ============================================================================
# LOAD CUSTOM ENGINE (with action_taken field)
# ============================================================================
import importlib
import sys
sys.path.insert(0, '/Users/nikhileshbelulkar/Documents/mit-poker-2026')

import custom_engine as engine_module

# Force reload the engine module to pick up the action_taken field
engine_module = importlib.reload(engine_module)

# Re-import the classes we need
from custom_engine import (
    RoundState, TerminalState, 
    FoldAction, CallAction, CheckAction, RaiseAction, DiscardAction,
    STARTING_STACK, BIG_BLIND, SMALL_BLIND
)

print("✓ Custom engine module loaded with action_taken field!")
print("  → RoundState now stores the action that led to each state")
print("  → This simplifies infoset history reconstruction")

✓ Custom engine module loaded with action_taken field!
  → RoundState now stores the action that led to each state
  → This simplifies infoset history reconstruction


---
## Step 1: Card Abstraction (Suit Isomorphism)


In [5]:
# ============================================================================
# CARD ABSTRACTION: SUIT ISOMORPHISM (Using canon_cards class)
# ============================================================================
# Reduces state space by ~75% by treating suit-equivalent hands as identical
#
# The canon_cards class from deep_CFR_vNB/canocalization_board_cards.py provides:
# - Rank-based suit canonicalization (highest card's suit becomes suit 0)
# - Automatic canonicalization on initialization
# - Helper methods: get_hand_str(), get_board_str()
# - Methods to add/remove cards while maintaining canonicalization
#
# Key idea: The actual suits don't matter, only the relationships.
# - If you have two spades, it doesn't matter if they're ♠ or ♥
# - What matters is: "I have two cards of the same suit"
#
# CRITICAL: Assigns suits by RANK order (highest card first)!
# This ensures the same canonicalization regardless of input order.
#
# Usage:
#   cards = canon_cards(hand, board)
#   hand_str = cards.get_hand_str()    # "As0,Ks0" (sorted)
#   board_str = cards.get_board_str()  # "2s1,Js0,Qs0" (sorted)


# Test card abstraction
print("✓ Card Abstraction (canon_cards class)")
print("\n" + "="*80)
print("TEST 1: Suit Isomorphism")
print("="*80)

# Example 1: Flush draw
from pkrbot import Card
test_hand1 = [Card("As"), Card("Ks")]  # A♠, K♠
test_board1 = [Card("Qs"), Card("Js"), Card("2h")]  # Q♠, J♠, 2♥

cards1 = canon_cards(test_hand1, test_board1)
print(f"Example 1 (A♠K♠ on Q♠J♠2♥):")
print(f"  Original hand:   {[str(c) for c in test_hand1]}")
print(f"  Original board:  {[str(c) for c in test_board1]}")
print(f"  Canonical hand:  {cards1.get_hand_str()}")
print(f"  Canonical board: {cards1.get_board_str()}")
print()

# Example 2: Same structure, different suits
test_hand2 = [Card("Ah"), Card("Kh")]  # A♥, K♥
test_board2 = [Card("Qh"), Card("Jh"), Card("2d")]  # Q♥, J♥, 2♦

cards2 = canon_cards(test_hand2, test_board2)
print(f"Example 2 (A♥K♥ on Q♥J♥2♦):")
print(f"  Original hand:   {[str(c) for c in test_hand2]}")
print(f"  Original board:  {[str(c) for c in test_board2]}")
print(f"  Canonical hand:  {cards2.get_hand_str()}")
print(f"  Canonical board: {cards2.get_board_str()}")
print()

if cards1.get_hand_str() == cards2.get_hand_str() and cards1.get_board_str() == cards2.get_board_str():
    print("✓ SUCCESS: Equivalent hands map to same canonical representation!")
    print("  → This reduces state space and speeds up learning!")
else:
    print("✗ ERROR: Equivalent hands have different canonical forms")

# ==============================================================================
# TEST 2: RANK-BASED CANONICALIZATION (Critical!)
# ==============================================================================
print("\n" + "="*80)
print("TEST 2: Rank-Based Canonicalization")
print("="*80)
print("Testing that card ORDER doesn't affect canonicalization:")
print()

# Scenario 1: Hand in one order
test_hand_A = [Card("Ks"), Card("Qh")]  # K♠ first
test_board_A = [Card("As"), Card("Jh")]  # A♠ first

# Scenario 2: Same cards, different order
test_hand_B = [Card("Qh"), Card("Ks")]  # Q♥ first
test_board_B = [Card("Jh"), Card("As")]  # J♥ first

cards_A = canon_cards(test_hand_A, test_board_A)
cards_B = canon_cards(test_hand_B, test_board_B)

print(f"Scenario A: Hand={[str(c) for c in test_hand_A]}, Board={[str(c) for c in test_board_A]}")
print(f"  → Canonical hand:  {cards_A.get_hand_str()}")
print(f"  → Canonical board: {cards_A.get_board_str()}")
print()

print(f"Scenario B: Hand={[str(c) for c in test_hand_B]}, Board={[str(c) for c in test_board_B]}")
print(f"  → Canonical hand:  {cards_B.get_hand_str()}")
print(f"  → Canonical board: {cards_B.get_board_str()}")
print()

if cards_A.get_hand_str() == cards_B.get_hand_str() and cards_A.get_board_str() == cards_B.get_board_str():
    print("✓ SUCCESS: Rank-based canonicalization works!")
    print("  → Same cards (different order) produce SAME canonical form!")
    print("  → This is crucial for proper state space reduction!")
else:
    print("✗ ERROR: Rank-based canonicalization FAILED!")
    print("  → Different canonical forms for the same cards!")
    print("  → This would cause duplicate states and slow learning!")


✓ Card Abstraction (canon_cards class)

TEST 1: Suit Isomorphism
Example 1 (A♠K♠ on Q♠J♠2♥):
  Original hand:   ['As', 'Ks']
  Original board:  ['Qs', 'Js', '2h']
  Canonical hand:  As0,Ks0
  Canonical board: 2s1,Js0,Qs0

Example 2 (A♥K♥ on Q♥J♥2♦):
  Original hand:   ['Ah', 'Kh']
  Original board:  ['Qh', 'Jh', '2d']
  Canonical hand:  As0,Ks0
  Canonical board: 2s1,Js0,Qs0

✓ SUCCESS: Equivalent hands map to same canonical representation!
  → This reduces state space and speeds up learning!

TEST 2: Rank-Based Canonicalization
Testing that card ORDER doesn't affect canonicalization:

Scenario A: Hand=['Ks', 'Qh'], Board=['As', 'Jh']
  → Canonical hand:  Ks0,Qs1
  → Canonical board: As0,Js1

Scenario B: Hand=['Qh', 'Ks'], Board=['Jh', 'As']
  → Canonical hand:  Ks0,Qs1
  → Canonical board: As0,Js1

✓ SUCCESS: Rank-based canonicalization works!
  → Same cards (different order) produce SAME canonical form!
  → This is crucial for proper state space reduction!


---
## Step 2: Information Set Abstraction (with Card Abstraction)

**Pseudocode Line 1**: `Initialize: ∀I ∈ I', ∀a ∈ A(I): r_I[a] ← s_I[a] ← 0`

An **information set** represents everything a player knows at a decision point:
- Their hole cards (canonicalized)
- Board cards (canonicalized)
- Betting history
- Current street

Now we'll use **canonical cards** from suit isomorphism to avoid duplicate states!

### 🔧 Critical Fixes Applied to `get_infoset()` Betting History

**Two major bugs were identified and fixed in the betting history extraction:**

#### **Bug 1: Chance Nodes Incorrectly Encoded as Player Discards**

**Problem:**
```python
# OLD (BROKEN):
if len(current.board) > len(prev.board):
    history.append('D')  # Marked ALL board growth as discards!
```

This incorrectly treated **chance events** (flop/turn/river dealing) as player discards:
- Flop dealt (0→2): 3 cards added → marked as 'D' ❌
- Turn dealt (3→4): 1 card added → marked as 'D' ❌
- Only actual player discards should be 'D' ✓

**Fix:**
```python
# NEW (FIXED):
if len(current.board) > len(prev.board):
    if prev.street in (2, 3) and current.street == prev.street:
        history.append('D')  # Only mark PLAYER discards
    # else: Chance event - don't append (MCCFR Line 4)
```

#### **Bug 2: Bet-to-Pot Ratio Used Current Street Pot Instead of Total Pot**

**Problem:**
```python
# OLD (BROKEN):
pot_before_bet = sum(prev.pips)  # Only current street!
```

Since `pips` reset to `[0, 0]` at street transitions (engine.py:192), this meant:
- First bet on flop/turn/river: `pot_before_bet = 0` ❌
- All bets miscategorized based on incorrect pot size

**Example**: 30-chip bet into 100-chip pot after preflop
- OLD: `pot = 0` → defaults to 'R' (medium) ❌
- NEW: `pot = 100` → ratio = 0.3 → 'r' (small) ✓

**Fix:**
```python
# NEW (FIXED):
pot_from_previous_streets = (2 * STARTING_STACK) - sum(prev.stacks)
pot_this_street = sum(prev.pips)
pot_before_bet = pot_from_previous_streets + pot_this_street
```

Now bet sizing is correctly calculated relative to the **total pot**, not just current street!

In [6]:
# ============================================================================
# INFORMATION SET ENCODING (WITH CARD ABSTRACTION)
# ============================================================================
# Maps RoundState → Canonical Information Set String for table lookup

def get_infoset(state: RoundState, player: int) -> str:
    """
    Convert a game state into a CANONICAL information set string.
    
    Key improvement: Uses suit isomorphism to map equivalent states together!
    - A♠K♠ on Q♠J♠2♣ → same infoset as A♥K♥ on Q♥J♥2♦
    
    An information set contains everything the player knows:
    - Their hand (canonicalized & sorted)
    - Board cards (canonicalized)
    - Betting history (sequence of actions)
    - Current street
    
    Args:
        state: Current RoundState
        player: Player index (0 or 1)
    
    Returns:
        Canonical string representation of information set
    """
    # ==================================================================
    # CARD ABSTRACTION: Use canon_cards class to canonicalize suits
    # ==================================================================
    # The canon_cards class handles:
    # - Rank-based suit canonicalization (highest card's suit → 0)
    # - Automatic sorting of hand and board cards
    # - This reduces state space by ~75% (suit isomorphism)
    # - Plus ~6× reduction from Markov property (card order doesn't matter)
    cards = canon_cards(state.hands[player], state.board)
    
    # Get sorted, canonical string representations
    # get_hand_str() returns sorted hand (e.g., "As0,Ks0")
    # get_board_str() returns sorted board (e.g., "2s1,Js0,Qs0")
    hand_str = cards.get_hand_str()
    board_str = cards.get_board_str() if state.board else ''
    
    street = state.street
    
    # ==================================================================
    # BETTING HISTORY: Extract action sequence from action_taken field
    # ==================================================================
    # SIMPLIFIED: Now we directly read action_taken instead of inferring!
    # - action_taken = FoldAction/CallAction/CheckAction/DiscardAction/RaiseAction
    # - action_taken = None for chance events (card dealing)
    history = []
    current = state
    while current.previous_state is not None:
        prev = current.previous_state
        
        # Use the action_taken field directly
        if hasattr(current, 'action_taken') and current.action_taken is not None:
            action = current.action_taken
            
            if isinstance(action, FoldAction):
                history.append('F')
            elif isinstance(action, CallAction):
                history.append('C')
            elif isinstance(action, CheckAction):
                history.append('X')
            elif isinstance(action, DiscardAction):
                history.append('D')
            elif isinstance(action, RaiseAction):
                # Calculate bet size relative to TOTAL pot
                pot_from_previous_streets = (2 * STARTING_STACK) - sum(prev.stacks)
                pot_this_street = sum(prev.pips)
                pot_before_bet = pot_from_previous_streets + pot_this_street
                
                bet_amount = max(current.pips) - max(prev.pips)
                
                if pot_before_bet > 0:
                    bet_to_pot_ratio = bet_amount / pot_before_bet
                    
                    # Bucket bet sizes into 3 categories
                    if bet_to_pot_ratio < 0.5:
                        history.append('r')  # Small bet
                    elif bet_to_pot_ratio < 1.0:
                        history.append('R')  # Medium bet
                    else:
                        history.append('B')  # Large bet
                else:
                    history.append('R')  # Default to medium
        # If action_taken is None, it's a chance event → don't add to history
        
        current = prev
    
    # Reverse to get chronological order
    history.reverse()
    history_str = ''.join(history[-20:])  # Keep last 20 actions to limit size
    
    # ==================================================================
    # COMBINE INTO CANONICAL INFOSET STRING
    # ==================================================================
    # Using canonical cards ensures equivalent states map to same string!
    infoset = f"S{street}|H:{hand_str}|B:{board_str}|A:{history_str}"
    return infoset


# Test information set encoding
print("✓ Information Set Encoding Defined (WITH card abstraction)")
print()
print("Testing with equivalent hands:")
print()

# Test with the same examples from card abstraction
from pkrbot import Card
test_hand1 = [Card("As"), Card("Ks")]  # A♠, K♠
test_board1 = [Card("Qs"), Card("Js"), Card("2h")]  # Q♠, J♠, 2♥
test_hand2 = [Card("Ah"), Card("Kh")]  # A♥, K♥  
test_board2 = [Card("Qh"), Card("Jh"), Card("2d")]  # Q♥, J♥, 2♦

# Create mock states (simplified for testing)
from collections import namedtuple
MockState = namedtuple('MockState', ['hands', 'board', 'street', 'pips', 'previous_state', 'action_taken'])

mock_state1 = MockState(
    hands=[test_hand1, []],
    board=test_board1,
    street=4,
    pips=[10, 10],
    previous_state=None,
    action_taken=None
)

mock_state2 = MockState(
    hands=[test_hand2, []],
    board=test_board2,
    street=4,
    pips=[10, 10],
    previous_state=None,
    action_taken=None
)

infoset1 = get_infoset(mock_state1, 0)
infoset2 = get_infoset(mock_state2, 0)

print(f"Infoset 1 (A♠K♠ on Q♠J♠2♥): {infoset1}")
print(f"Infoset 2 (A♥K♥ on Q♥J♥2♦): {infoset2}")
print()

if infoset1 == infoset2:
    print("✓ SUCCESS: Equivalent hands produce SAME infoset!")
    print("  → State space reduced ~75%")
    print("  → Learning will be ~4× faster!")
else:
    print("✗ Different infosets (unexpected)")

print("\n" + "="*80)
print("TEST 2: Markov Property - Card Ordering Across Streets")
print("="*80)

# Test that different orderings of board cards produce the same infoset
# Scenario A: Flop[Qs,Js,2h] Turn[5d] River[7c]
test_board_A = [Card("Qs"), Card("Js"), Card("2h"), Card("5d"), Card("7c")]

# Scenario B: Flop[5d,7c,Qs] Turn[Js] River[2h] (same cards, different order)
test_board_B = [Card("5d"), Card("7c"), Card("Qs"), Card("Js"), Card("2h")]

mock_state_A = MockState(
    hands=[[Card("As"), Card("Ks")], []],
    board=test_board_A,
    street=6,  # River
    pips=[50, 50],
    previous_state=None,
    action_taken=None
)

mock_state_B = MockState(
    hands=[[Card("As"), Card("Ks")], []],
    board=test_board_B,
    street=6,  # River
    pips=[50, 50],
    previous_state=None,
    action_taken=None
)

infoset_A = get_infoset(mock_state_A, 0)
infoset_B = get_infoset(mock_state_B, 0)

print(f"Board A: {[str(c) for c in test_board_A]}")
print(f"Infoset A: {infoset_A}")
print()
print(f"Board B: {[str(c) for c in test_board_B]}")
print(f"Infoset B: {infoset_B}")
print()

if infoset_A == infoset_B:
    print("✓ SUCCESS: Same board cards (different order) → SAME infoset!")
    print("  → Markov property verified!")
    print("  → State space reduced by another ~5-6× (board permutations)")
    print("  → Total abstraction: ~75% (suits) × ~6× (orderings) = ~90-95% reduction!")
else:
    print("✗ FAILED: Different infosets (should be the same!)")
    print(f"  Difference: {set(infoset_A) - set(infoset_B)}")

✓ Information Set Encoding Defined (WITH card abstraction)

Testing with equivalent hands:

Infoset 1 (A♠K♠ on Q♠J♠2♥): S4|H:As0,Ks0|B:2s1,Js0,Qs0|A:
Infoset 2 (A♥K♥ on Q♥J♥2♦): S4|H:As0,Ks0|B:2s1,Js0,Qs0|A:

✓ SUCCESS: Equivalent hands produce SAME infoset!
  → State space reduced ~75%
  → Learning will be ~4× faster!

TEST 2: Markov Property - Card Ordering Across Streets
Board A: ['Qs', 'Js', '2h', '5d', '7c']
Infoset A: S6|H:As0,Ks0|B:2s3,5s2,7s1,Js0,Qs0|A:

Board B: ['5d', '7c', 'Qs', 'Js', '2h']
Infoset B: S6|H:As0,Ks0|B:2s3,5s2,7s1,Js0,Qs0|A:

✓ SUCCESS: Same board cards (different order) → SAME infoset!
  → Markov property verified!
  → State space reduced by another ~5-6× (board permutations)
  → Total abstraction: ~75% (suits) × ~6× (orderings) = ~90-95% reduction!


---
## Step 3: Action Abstraction & Encoding

**Challenge**: Continuous action space (can raise any amount from min to max)

**Why we need this**: In **External Sampling MCCFR**, we need to actually **choose and execute** specific bet sizes when we sample actions!

**Solution**: Discretize into buckets to make state space tractable

### 🎯 Action Abstraction Strategy (for External Sampling):

1. **Raises**: Exactly **3 discrete sizes** based on engine's legal bounds
   - **MIN**: `state.raise_bounds()[0]` - Minimum legal raise
   - **MID**: `(min_raise + max_raise) // 2` - Average/midpoint
   - **MAX**: `state.raise_bounds()[1]` - Maximum legal raise (often all-in)
   
   Example: If min=10, max=100 → Actions: [Raise(10), Raise(55), Raise(100)]

2. **Discards**: All cards in hand (typically 2-3 options after flop)
   - `DiscardAction(0)`, `DiscardAction(1)`, `DiscardAction(2)` (for 3-card hands)

3. **Check/Call/Fold**: Single option each (no abstraction needed)

In [7]:
# ============================================================================
# ACTION ENCODING
# ============================================================================
# Maps actions ↔ indices for array-based regret/strategy storage

def action_to_key(action, state=None, active_player=None) -> str:
    """
    Convert an action object to a string key for dictionary lookup.
    
    CRITICAL FIX: For DiscardActions, we encode by CANONICAL CARD VALUE,
    not position index! This ensures same cards map to same actions 
    regardless of hand order.
    
    Args:
        action: One of FoldAction, CallAction, CheckAction, RaiseAction, DiscardAction
        state: Current RoundState (REQUIRED for DiscardAction to get canonical card)
        active_player: Player index (REQUIRED for DiscardAction)
    
    Returns:
        String key representing the action
        
    Example:
        Hand=[Ks,Qh,2c] → Canonical=[2s0,Ks0,Qs1] (sorted)
        DiscardAction(0) → "DISCARD_Ks0" (canonical card at position 0)
        
        Hand=[2c,Ks,Qh] → Canonical=[2s0,Ks0,Qs1] (same after sorting!)
        DiscardAction(1) → "DISCARD_Ks0" (same action! ✓)
    """
    if isinstance(action, FoldAction):
        return "FOLD"
    elif isinstance(action, CallAction):
        return "CALL"
    elif isinstance(action, CheckAction):
        return "CHECK"
    elif isinstance(action, RaiseAction):
        # Bucket raise amounts for abstraction
        return f"RAISE_{action.amount}"
    elif isinstance(action, DiscardAction):
        # CRITICAL FIX: Encode by CANONICAL CARD VALUE, not position!
        # This ensures same cards map to same actions regardless of hand order
        if state is not None and active_player is not None:
            # Use canon_cards class to get canonical hand
            cards = canon_cards(state.hands[active_player], state.board)
            canonical_hand = cards.canonical_hand
            # Find which canonical card corresponds to this position
            canonical_card = canonical_hand[action.card]
            return f"DISCARD_{canonical_card}"
        else:
            # Fallback (shouldn't happen in normal flow)
            return f"DISCARD_{action.card}"
    elif isinstance(action, type):
        # Handle action types (not instances)
        if action == FoldAction:
            return "FOLD"
        elif action == CallAction:
            return "CALL"
        elif action == CheckAction:
            return "CHECK"
        elif action == RaiseAction:
            return "RAISE"
        elif action == DiscardAction:
            return "DISCARD"
    return str(action)


def get_legal_actions_list(state: RoundState) -> List:
    """
    Get list of legal actions with concrete values (ACTION ABSTRACTION).
    
    CRITICAL for External Sampling MCCFR: We need to actually EXECUTE specific
    bet sizes when sampling actions, not ranges!
    
    Action Abstraction Strategy:
    - Raises: Exactly 3 discrete sizes (MIN, MID, MAX) from engine's raise_bounds()
    - Discards: All cards in hand (no abstraction)
    - Check/Call/Fold: Single action (no abstraction)
    
    Args:
        state: Current RoundState
    
    Returns:
        List of concrete action instances ready to execute
        Example: [FoldAction(), CallAction(), RaiseAction(10), RaiseAction(55), RaiseAction(100)]
    """
    legal_action_types = state.legal_actions()
    actions = []
    
    active = state.button % 2
    
    for action_type in legal_action_types:
        if action_type == RaiseAction:
            # ACTION ABSTRACTION: Discretize continuous raise space into 3 sizes
            min_raise, max_raise = state.raise_bounds()
            
            # Exactly 3 raise sizes for external sampling:
            # 1. MIN: Minimum legal raise (small bet)
            # 2. MID: Average between min and max (medium bet)
            # 3. MAX: Maximum legal raise, often all-in (large bet)
            raise_sizes = [
                min_raise,                          # MIN
                (min_raise + max_raise) // 2,       # MID (average)
                max_raise                           # MAX
            ]
            # Remove duplicates (e.g., if min==max, we only have 1 size)
            raise_sizes = sorted(list(set(raise_sizes)))
            
            for size in raise_sizes:
                actions.append(RaiseAction(size))
                
        elif action_type == DiscardAction:
            # All cards in hand can be discarded
            for card_idx in range(len(state.hands[active])):
                actions.append(DiscardAction(card_idx))
        else:
            # Fold, Call, Check - just add the type
            actions.append(action_type())
    
    return actions


print("✓ Action encoding functions defined")
print("  - action_to_key(): Maps action → string key")
print("  - get_legal_actions_list(): Gets concrete legal actions with values")

✓ Action encoding functions defined
  - action_to_key(): Maps action → string key
  - get_legal_actions_list(): Gets concrete legal actions with values


### 🔧 Critical Fix: Canonical Discard Action Encoding

**Problem Identified**: Discard actions were encoded by POSITION INDEX, not canonical card value!

**Why This Breaks MCCFR:**
```
Scenario A: Hand = [Ks, Qh, 2c] → Canonical = [2s0, Ks0, Qs1] (sorted)
  DiscardAction(0) → "DISCARD_0" (discards Ks)

Scenario B: Hand = [2c, Ks, Qh] → Canonical = [2s0, Ks0, Qs1] (SAME!)
  DiscardAction(1) → "DISCARD_1" (discards Ks)
  
Problem: Same information set, but "DISCARD_0" ≠ "DISCARD_1"!
  → MCCFR treats discarding the SAME card as DIFFERENT actions
  → Breaks state space abstraction
  → Slows learning dramatically
```

**The Fix**: Encode discard actions by **CANONICAL CARD VALUE**
```python
# OLD (BROKEN):
def action_to_key(action):
    elif isinstance(action, DiscardAction):
        return f"DISCARD_{action.card}"  # position index!

# NEW (FIXED):
def action_to_key(action, state, active_player):
    elif isinstance(action, DiscardAction):
        canonical_hand, _ = canonicalize_cards(state.hands[active_player], state.board)
        canonical_card = canonical_hand[action.card]
        return f"DISCARD_{canonical_card}"  # canonical card value!
```

**Result**: Same canonical card always produces the same action key, regardless of hand order! ✓

In [8]:
# ==============================================================================
# TEST: Verify Canonical Discard Action Encoding
# ==============================================================================
print("="*80)
print("TEST: Canonical Discard Action Encoding (Critical Fix!)")
print("="*80)
print()

# Create two test scenarios with SAME cards but DIFFERENT order
from collections import namedtuple
MockState = namedtuple('MockState', ['hands', 'board', 'street', 'button'])

# Scenario A: Hand in order [Ks, Qh, 2c]
test_hand_A = [Card("Ks"), Card("Qh"), Card("2c")]
test_board_A = [Card("As"), Card("Jh"), Card("5d")]

mock_state_A = MockState(
    hands=[test_hand_A, []],
    board=test_board_A,
    street=2,
    button=1
)

# Scenario B: Same cards, different order [2c, Ks, Qh]
test_hand_B = [Card("2c"), Card("Ks"), Card("Qh")]
test_board_B = [Card("As"), Card("Jh"), Card("5d")]

mock_state_B = MockState(
    hands=[test_hand_B, []],
    board=test_board_B,
    street=2,
    button=1
)

print(f"Scenario A: Hand = {[str(c) for c in test_hand_A]}")
cards_test_A = canon_cards(test_hand_A, test_board_A)
print(f"  Canonical hand: {cards_test_A.canonical_hand}")
print()

print(f"Scenario B: Hand = {[str(c) for c in test_hand_B]}")
cards_test_B = canon_cards(test_hand_B, test_board_B)
print(f"  Canonical hand: {cards_test_B.canonical_hand}")
print()

# Test discard action encoding
print("Discard Action Encodings:")
print("-" * 80)

# Scenario A: Discard each card
for i in range(3):
    action = DiscardAction(i)
    action_key = action_to_key(action, mock_state_A, 0)
    card_at_pos = test_hand_A[i]
    print(f"Scenario A - DiscardAction({i}) [card={card_at_pos}] → '{action_key}'")

print()

# Scenario B: Discard each card
for i in range(3):
    action = DiscardAction(i)
    action_key = action_to_key(action, mock_state_B, 0)
    card_at_pos = test_hand_B[i]
    print(f"Scenario B - DiscardAction({i}) [card={card_at_pos}] → '{action_key}'")

print()
print("="*80)

# Verify that discarding THE SAME CANONICAL CARD produces THE SAME key
action_A_discard_K = DiscardAction(0)  # Ks at position 0 in hand A
action_B_discard_K = DiscardAction(1)  # Ks at position 1 in hand B

key_A = action_to_key(action_A_discard_K, mock_state_A, 0)
key_B = action_to_key(action_B_discard_K, mock_state_B, 0)

print(f"Discarding K♠:")
print(f"  Scenario A (pos 0): {key_A}")
print(f"  Scenario B (pos 1): {key_B}")

if key_A == key_B:
    print("\n✓ SUCCESS: Same canonical card → SAME action key!")
    print("  → MCCFR will correctly learn that these are the same action!")
    print("  → State space reduced, learning accelerated!")
else:
    print("\n✗ FAILED: Different action keys for same card!")
    print("  → This would break MCCFR learning!")

print("="*80)

TEST: Canonical Discard Action Encoding (Critical Fix!)

Scenario A: Hand = ['Ks', 'Qh', '2c']
  Canonical hand: ['Ks0', 'Qs1', '2s3']

Scenario B: Hand = ['2c', 'Ks', 'Qh']
  Canonical hand: ['2s3', 'Ks0', 'Qs1']

Discard Action Encodings:
--------------------------------------------------------------------------------
Scenario A - DiscardAction(0) [card=Ks] → 'DISCARD_Ks0'
Scenario A - DiscardAction(1) [card=Qh] → 'DISCARD_Qs1'
Scenario A - DiscardAction(2) [card=2c] → 'DISCARD_2s3'

Scenario B - DiscardAction(0) [card=2c] → 'DISCARD_2s3'
Scenario B - DiscardAction(1) [card=Ks] → 'DISCARD_Ks0'
Scenario B - DiscardAction(2) [card=Qh] → 'DISCARD_Qs1'

Discarding K♠:
  Scenario A (pos 0): DISCARD_Ks0
  Scenario B (pos 1): DISCARD_Ks0

✓ SUCCESS: Same canonical card → SAME action key!
  → MCCFR will correctly learn that these are the same action!
  → State space reduced, learning accelerated!


---
## Step 4: Regret Matching

**Pseudocode Line 6**: `σ(I) ← RegretMatching(r_I)`

Regret matching converts cumulative regrets into a strategy (probability distribution over actions):
- Take positive regrets only (clamp negatives to 0)
- Normalize to sum to 1
- If all regrets ≤ 0, use uniform distribution over legal actions

In [9]:
# ============================================================================
# REGRET MATCHING (Pseudocode Line 6)
# ============================================================================
# σ(I) ← RegretMatching(r_I)
# Converts cumulative regrets into a probability distribution (strategy)

def regret_matching(regrets: Dict[str, float], actions: List, state=None, active_player=None) -> Dict[str, float]:
    """
    Compute strategy from regrets using regret matching.
    
    Algorithm:
    1. Take max(regret, 0) for each action  [positive regrets only]
    2. Sum all positive regrets
    3. If sum > 0: normalize to get probabilities
    4. If sum = 0: uniform distribution over all actions
    
    Args:
        regrets: Dictionary mapping action_key → cumulative regret
        actions: List of legal action instances
        state: Current RoundState (needed for DiscardAction encoding)
        active_player: Player index (needed for DiscardAction encoding)
    
    Returns:
        Dictionary mapping action_key → probability
    """
    strategy = {}
    action_keys = [action_to_key(a, state, active_player) for a in actions]
    
    # Step 1: Clamp to positive regrets only
    positive_regrets = {key: max(regrets.get(key, 0.0), 0.0) for key in action_keys}
    
    # Step 2: Sum positive regrets
    sum_positive = sum(positive_regrets.values())
    
    # Step 3 & 4: Normalize or use uniform
    if sum_positive > 0:
        # Normalize: probability ∝ positive regret
        strategy = {key: positive_regrets[key] / sum_positive for key in action_keys}
    else:
        # Uniform distribution
        uniform_prob = 1.0 / len(action_keys) if action_keys else 0.0
        strategy = {key: uniform_prob for key in action_keys}
    
    return strategy


# Example of regret matching
example_regrets = {'FOLD': -10.0, 'CALL': 5.0, 'RAISE_100': 15.0}
example_actions = [FoldAction(), CallAction(), RaiseAction(100)]
example_strategy = regret_matching(example_regrets, example_actions)

print("✓ Regret Matching Function Defined")
print("\nExample:")
print(f"  Regrets: {example_regrets}")
print(f"  Strategy: {example_strategy}")
print(f"  → Fold gets 0% (negative regret ignored)")
print(f"  → Call gets {example_strategy['CALL']*100:.1f}% (5 / (5+15))")
print(f"  → Raise gets {example_strategy['RAISE_100']*100:.1f}% (15 / (5+15))")

✓ Regret Matching Function Defined

Example:
  Regrets: {'FOLD': -10.0, 'CALL': 5.0, 'RAISE_100': 15.0}
  Strategy: {'FOLD': 0.0, 'CALL': 0.25, 'RAISE_100': 0.75}
  → Fold gets 0% (negative regret ignored)
  → Call gets 25.0% (5 / (5+15))
  → Raise gets 75.0% (15 / (5+15))


---
## Step 5: Initialize Regret and Strategy Tables

**Pseudocode Line 1**: `Initialize: ∀I ∈ Z, ∀a ∈ A(I): r_I[a] ← s_I[a] ← 0`

We use nested dictionaries:
- Outer key: Information set string
- Inner key: Action string  
- Value: Cumulative regret (r_I) or cumulative strategy (s_I)

In [10]:
# ============================================================================
# INITIALIZE TABLES (Pseudocode Line 1)
# ============================================================================
# Initialize: ∀I ∈ Z, ∀a ∈ A(I): r_I[a] ← s_I[a] ← 0

# Regret table: r_I[a] = cumulative regret for action a at infoset I
regret_table = defaultdict(lambda: defaultdict(float))

# Strategy table: s_I[a] = cumulative strategy for action a at infoset I  
strategy_table = defaultdict(lambda: defaultdict(float))

print("✓ Tables Initialized")
print("  regret_table[infoset][action] = cumulative regret")
print("  strategy_table[infoset][action] = cumulative strategy")
print("  (Using defaultdict with automatic 0.0 initialization)")

✓ Tables Initialized
  regret_table[infoset][action] = cumulative regret
  strategy_table[infoset][action] = cumulative strategy
  (Using defaultdict with automatic 0.0 initialization)


---
## Step 6: External Sampling MCCFR Algorithm

**Pseudocode Lines 2-21**: The core recursive CFR traversal

This is the heart of the algorithm. It recursively traverses the game tree:

**Line 3**: If terminal, return utility  
**Line 4**: If chance node, sample action and recurse  
**Lines 5-15**: If traversing player's turn:
  - Compute value of each action (line 10)
  - Accumulate weighted values (line 11)
  - Compute regrets (line 13)
  - Update regret table (line 14)
  - Return expected value (line 15)

**Lines 16-21**: If opponent's turn:
  - Sample action from current strategy (line 17)
  - Recurse to get value (line 18)
  - Update strategy table (line 20)
  - Return value (line 21)

In [11]:
# ============================================================================
# EXTERNAL SAMPLING (Pseudocode Lines 2-21)
# ============================================================================
# ExternalSampling(h, i): Recursive CFR traversal

def external_sampling(state: Union[RoundState, TerminalState], 
                     traversing_player: int,
                     infoset_cache: dict = None) -> float:
    """
    External sampling MCCFR traversal.
    
    This function implements Algorithm 4 from the pseudocode exactly.
    
    Args:
        state: Current game state (RoundState or TerminalState)
        traversing_player: Player index whose regrets we're updating (0 or 1)
        infoset_cache: Dict caching state_id -> infoset string (avoids recomputation)
    
    Returns:
        Utility value for the traversing player at this node
    """
    
    # Initialize cache if not provided
    if infoset_cache is None:
        infoset_cache = {}
    
    # ========================================================================
    # LINE 3: if h ∈ Z then return u_i(h)
    # ========================================================================
    # Terminal state: return utility
    if isinstance(state, TerminalState):
        return float(state.deltas[traversing_player])
    
    # ========================================================================
    # LINE 4: if P(h) = c then sample a' and return ExternalSampling(ha', i)
    # ========================================================================
    # Chance node: In our poker game, chance is handled by the deck
    # The initial deal is chance, but subsequent states are player decisions
    # We don't explicitly sample here as the engine handles card dealing
    
    # ========================================================================
    # LINE 5: Let I be the information set containing h
    # ========================================================================
    active_player = state.button % 2
    
    # OPTIMIZATION: Cache infoset lookups to avoid recomputing history
    state_id = id(state)
    if state_id in infoset_cache:
        infoset = infoset_cache[state_id]
    else:
        infoset = get_infoset(state, active_player)
        infoset_cache[state_id] = infoset
    
    # Get legal actions at this decision point
    legal_actions = get_legal_actions_list(state)
    
    if not legal_actions:
        # No legal actions (shouldn't happen, but handle gracefully)
        return 0.0
    
    # ========================================================================
    # LINE 6: σ(I) ← RegretMatching(r_I)
    # ========================================================================
    # Get current strategy from regrets via regret matching
    strategy = regret_matching(regret_table[infoset], legal_actions, state, active_player)
    
    # ========================================================================
    # LINE 7: if P(I) = i then
    # ========================================================================
    # Check if it's the traversing player's turn
    if active_player == traversing_player:
        # ====================================================================
        # LINES 8-15: Traversing player's node
        # ====================================================================
        
        # LINE 8: Let u be an array indexed by actions and u_σ ← 0
        action_values = {}  # u[a] for each action
        node_value = 0.0    # u_σ (expected value of node)
        
        # LINE 9-11: for a ∈ A(I) do
        for action in legal_actions:
            action_key = action_to_key(action, state, active_player)
            
            # LINE 10: u[a] ← ExternalSampling(ha, i)
            # Recurse to get value of taking this action
            next_state = state.proceed(action)
            action_values[action_key] = external_sampling(next_state, traversing_player, infoset_cache)
            
            # LINE 11: u_σ ← u_σ + σ(I, a) · u[a]
            # Accumulate weighted value
            node_value += strategy[action_key] * action_values[action_key]
        
        # LINE 12-14: for a ∈ A(I) do
        for action in legal_actions:
            action_key = action_to_key(action, state, active_player)
            
            # LINE 13: By Equation 4.20, compute r̃(I, a) ← u[a] - u_σ
            # Regret = value of action - expected value of node
            regret = action_values[action_key] - node_value
            
            # LINE 14: r_I[a] ← r_I[a] + r̃(I, a)
            # Update cumulative regret
            regret_table[infoset][action_key] += regret
        
        # LINE 15: return u_σ
        return node_value
    
    else:
        # ====================================================================
        # LINES 16-21: Opponent's node (not traversing player)
        # ====================================================================
        
        # LINE 17: Sample action a' from σ(I)
        # Sample an action according to current strategy
        action_keys = [action_to_key(a, state, active_player) for a in legal_actions]
        probs = [strategy[key] for key in action_keys]
        sampled_action = random.choices(legal_actions, weights=probs, k=1)[0]
        sampled_key = action_to_key(sampled_action, state, active_player)
        
        # LINE 18: u ← ExternalSampling(ha', i)
        # Recurse with sampled action
        next_state = state.proceed(sampled_action)
        value = external_sampling(next_state, traversing_player, infoset_cache)
        
        # LINE 19-20: for a ∈ A(I) do: s_I[a] ← s_I[a] + σ(I, a)
        # Update cumulative strategy for all legal actions
        for action in legal_actions:
            action_key = action_to_key(action, state, active_player)
            strategy_table[infoset][action_key] += strategy[action_key]
        
        # LINE 21: return u
        return value


print("✓ External Sampling MCCFR Function Defined")
print("  Implements Algorithm 4: External Sampling with Stochastically-Weighted Averaging")
print("  - Lines 3-4: Terminal/Chance nodes")
print("  - Lines 5-15: Traversing player (update regrets)")
print("  - Lines 16-21: Opponent (sample action, update strategy)")

✓ External Sampling MCCFR Function Defined
  Implements Algorithm 4: External Sampling with Stochastically-Weighted Averaging
  - Lines 3-4: Terminal/Chance nodes
  - Lines 5-15: Traversing player (update regrets)
  - Lines 16-21: Opponent (sample action, update strategy)


---
## Step 7: Game Initialization

Helper function to create a new game state with random cards dealt.

In [12]:
# ============================================================================
# GAME INITIALIZATION
# ============================================================================
# Create initial game states for training

# Import Deck class from engine
sys.path.append(os.path.join(os.path.dirname(os.path.abspath('__file__')), '../..'))
from pkrbot import Deck

def create_initial_state() -> RoundState:
    """
    Create a new game state with cards dealt.
    
    This represents the start of a poker hand:
    - Both players have starting stacks
    - Small blind posted by button (player 0)
    - Big blind posted by player 1
    - **THREE cards dealt to each player** (MIT 2026 variant)
    - After flop, each player discards 1 card face-up to the board
    - Button = 0 (player 0 acts first preflop)
    
    Returns:
        RoundState at the start of preflop betting
    """
    # Create and shuffle deck
    deck = Deck()
    deck.shuffle()
    
    # Deal 3 cards to each player (MIT 2026 variant: start with 3, discard 1 after flop)
    hands = [deck.deal(3), deck.deal(3)]
    
    # Initial state: preflop, button at 0 (small blind acts first)
    # Pips: [SMALL_BLIND, BIG_BLIND] (blinds already posted)
    # Stacks: reduced by blind amounts
    # Street 0 = preflop
    # Board is empty preflop
    initial_state = RoundState(
        button=0,                                    # Player 0 acts first preflop
        street=0,                                    # Street 0 = preflop
        pips=[SMALL_BLIND, BIG_BLIND],              # Blinds posted
        stacks=[STARTING_STACK - SMALL_BLIND,       # Stacks reduced by blinds
                STARTING_STACK - BIG_BLIND],
        hands=hands,                                 # Hole cards
        deck=deck,                                   # Remaining deck
        board=[],                                    # No board cards yet
        previous_state=None,                         # Start of game
        action_taken=None                            # No action yet (initial state)
    )
    
    return initial_state


# Test game initialization
test_state = create_initial_state()
print("✓ Game Initialization Function Defined")
print(f"\nExample initial state:")
print(f"  Street: {test_state.street} (0=preflop)")
print(f"  Button: {test_state.button} (active player)")
print(f"  Pips: {test_state.pips} (blinds posted)")
print(f"  Stacks: {test_state.stacks}")
print(f"  Hand 0: {[str(c) for c in test_state.hands[0]]}")
print(f"  Hand 1: {[str(c) for c in test_state.hands[1]]}")
print(f"  Board: {test_state.board} (empty preflop)")

✓ Game Initialization Function Defined

Example initial state:
  Street: 0 (0=preflop)
  Button: 0 (active player)
  Pips: [1, 2] (blinds posted)
  Stacks: [399, 398]
  Hand 0: ['Jh', '7d', '3h']
  Hand 1: ['2s', '4d', 'Js']
  Board: [] (empty preflop)


---
## Step 8: Training Loop

Run multiple iterations of MCCFR:
1. Create a new game (random cards)
2. Traverse from Player 0's perspective
3. Traverse from Player 1's perspective
4. Repeat for N iterations

The strategy converges to Nash equilibrium as iterations → ∞

In [13]:
# ============================================================================
# TRAINING LOOP
# ============================================================================
# Run MCCFR for multiple iterations

def train_mccfr(num_iterations: int, verbose: bool = True):
    """
    Train MCCFR agent by running external sampling traversals.
    
    Each iteration:
    1. Deal a new random hand
    2. Run external sampling for Player 0 (updates P0's regrets)
    3. Run external sampling for Player 1 (updates P1's regrets)
    
    After training, strategy_table contains the average strategy,
    which converges to Nash equilibrium.
    
    Args:
        num_iterations: Number of hands to play
        verbose: Whether to print progress
    """
    import time
    
    utilities = []
    iteration_times = []  # Track timing per iteration
    start_time = time.time()
    
    for iteration in range(num_iterations):
        iter_start = time.time()  # Start timing this iteration
        
        # Create new game with random cards
        initial_state = create_initial_state()
        
        # Traverse from Player 0's perspective
        # This updates regrets for Player 0's information sets
        utility_p0 = external_sampling(initial_state, traversing_player=0)
        
        # Traverse from Player 1's perspective  
        # This updates regrets for Player 1's information sets
        utility_p1 = external_sampling(initial_state, traversing_player=1)
        
        utilities.append((utility_p0, utility_p1))
        
        iter_time = time.time() - iter_start  # Calculate iteration time
        iteration_times.append(iter_time)
        
        # Print progress
        if verbose and (iteration + 1) % 100 == 0:
            avg_p0 = np.mean([u[0] for u in utilities[-100:]])
            avg_p1 = np.mean([u[1] for u in utilities[-100:]])
            avg_iter_time = np.mean(iteration_times[-100:])
            total_elapsed = time.time() - start_time
            estimated_remaining = avg_iter_time * (num_iterations - iteration - 1)
            
            print(f"Iteration {iteration + 1}/{num_iterations}")
            print(f"  Avg utility P0: {avg_p0:.2f} chips/hand")
            print(f"  Avg utility P1: {avg_p1:.2f} chips/hand")
            print(f"  Infosets learned: {len(regret_table)}")
            print(f"  Avg time/iter: {avg_iter_time:.3f}s ({1/avg_iter_time:.1f} iter/s)")
            print(f"  Elapsed: {total_elapsed:.1f}s | ETA: {estimated_remaining:.1f}s")
    
    total_time = time.time() - start_time
    if verbose:
        print(f"\n✓ Training Complete!")
        print(f"  Total iterations: {num_iterations}")
        print(f"  Total time: {total_time:.1f}s ({total_time/60:.1f} min)")
        print(f"  Avg time/iteration: {np.mean(iteration_times):.3f}s")
        print(f"  Information sets: {len(regret_table)}")
        print(f"  Strategy entries: {len(strategy_table)}")
    
    return utilities


print("✓ Training Loop Defined (with timing & ETA)")
print("  Call train_mccfr(num_iterations) to start training")

✓ Training Loop Defined (with timing & ETA)
  Call train_mccfr(num_iterations) to start training


---
## Step 9: Strategy Extraction

After training, we need to extract the learned strategy from the strategy table.

The **average strategy** (stored in `strategy_table`) converges to Nash equilibrium, NOT the instantaneous regret-matching strategy.

In [14]:
# ============================================================================
# STRATEGY EXTRACTION
# ============================================================================
# Get the learned average strategy (converges to Nash equilibrium)

def get_average_strategy(infoset: str, actions: List, state=None, active_player=None) -> Dict[str, float]:
    """
    Extract the average strategy for an information set.
    
    The average strategy is computed from strategy_table (cumulative strategies),
    NOT from regret_table. This is the key insight: the AVERAGE strategy
    converges to Nash equilibrium, not the instantaneous regret-matching strategy.
    
    Args:
        infoset: Information set string
        actions: List of legal actions
        state: Current RoundState (needed for DiscardAction encoding)
        active_player: Player index (needed for DiscardAction encoding)
    
    Returns:
        Dictionary mapping action_key → probability
    """
    strategy = {}
    action_keys = [action_to_key(a, state, active_player) for a in actions]
    
    # Sum cumulative strategies
    total = sum(strategy_table[infoset][key] for key in action_keys)
    
    if total > 0:
        # Normalize cumulative strategies to get average strategy
        strategy = {key: strategy_table[infoset][key] / total for key in action_keys}
    else:
        # If no data, use uniform distribution
        uniform_prob = 1.0 / len(action_keys) if action_keys else 0.0
        strategy = {key: uniform_prob for key in action_keys}
    
    return strategy


def select_action(state: RoundState, player: int, use_average: bool = True) -> any:
    """
    Select an action for a player at a given state using the learned strategy.
    
    Args:
        state: Current game state
        player: Player index
        use_average: If True, use average strategy (for final policy)
                    If False, use current regret-matching strategy (for exploration)
    
    Returns:
        Action to take
    """
    infoset = get_infoset(state, player)
    legal_actions = get_legal_actions_list(state)
    
    if not legal_actions:
        return None
    
    if use_average:
        # Use average strategy (Nash equilibrium approximation)
        strategy = get_average_strategy(infoset, legal_actions, state, player)
    else:
        # Use current regret-matching strategy
        strategy = regret_matching(regret_table[infoset], legal_actions, state, player)
    
    # Sample action according to strategy
    action_keys = [action_to_key(a, state, player) for a in legal_actions]
    probs = [strategy[key] for key in action_keys]
    selected_action = random.choices(legal_actions, weights=probs, k=1)[0]
    
    return selected_action


print("✓ Strategy Extraction Functions Defined")
print("  - get_average_strategy(): Extract Nash equilibrium approximation")
print("  - select_action(): Choose action using learned policy")

✓ Strategy Extraction Functions Defined
  - get_average_strategy(): Extract Nash equilibrium approximation
  - select_action(): Choose action using learned policy


In [15]:
# ============================================================================
# TABLE INSPECTION UTILITIES
# ============================================================================
# Helper functions to display and analyze learned strategies and regrets

def print_strategy_table(num_infosets=10, sort_by='frequency'):
    """
    Display the learned average strategy in a readable format.
    
    Args:
        num_infosets: Number of infosets to display
        sort_by: 'frequency' (by visit count) or 'regret' (by total regret magnitude)
    """
    print("="*80)
    print(f"STRATEGY TABLE (Top {num_infosets} Infosets)")
    print("="*80)
    print()
    
    if not strategy_table:
        print("No strategy data yet. Train first!")
        return
    
    # Calculate visit counts (sum of cumulative strategy across all actions)
    infoset_visits = {
        infoset: sum(actions.values())
        for infoset, actions in strategy_table.items()
    }
    
    # Sort infosets by visit count
    if sort_by == 'frequency':
        sorted_infosets = sorted(infoset_visits.items(), key=lambda x: x[1], reverse=True)
    else:  # sort by regret magnitude
        regret_magnitudes = {
            infoset: sum(abs(r) for r in regret_table[infoset].values())
            for infoset in strategy_table.keys()
        }
        sorted_infosets = sorted(regret_magnitudes.items(), key=lambda x: x[1], reverse=True)
        sorted_infosets = [(infoset, infoset_visits[infoset]) for infoset, _ in sorted_infosets]
    
    # Display top infosets
    for i, (infoset, visits) in enumerate(sorted_infosets[:num_infosets], 1):
        print(f"Infoset {i} (visited {visits:.1f} times):")
        print(f"  {infoset}")
        print()
        
        # Get average strategy for this infoset
        cumulative_strategy = strategy_table[infoset]
        total = sum(cumulative_strategy.values())
        
        if total > 0:
            avg_strategy = {action: count / total for action, count in cumulative_strategy.items()}
            
            # Sort actions by probability
            sorted_actions = sorted(avg_strategy.items(), key=lambda x: x[1], reverse=True)
            
            print("  Average Strategy:")
            for action, prob in sorted_actions:
                print(f"    {action:20s}: {prob*100:5.1f}%")
        else:
            print("  No strategy data (not visited during opponent's turns)")
        
        print()
    
    print("="*80)


def print_regret_table(num_infosets=10):
    """
    Display cumulative regrets for debugging.
    
    Args:
        num_infosets: Number of infosets to display
    """
    print("="*80)
    print(f"REGRET TABLE (Top {num_infosets} Infosets by Total Regret)")
    print("="*80)
    print()
    
    if not regret_table:
        print("No regret data yet. Train first!")
        return
    
    # Calculate total regret magnitude for each infoset
    regret_magnitudes = {
        infoset: sum(abs(r) for r in actions.values())
        for infoset, actions in regret_table.items()
    }
    
    # Sort by total regret magnitude
    sorted_infosets = sorted(regret_magnitudes.items(), key=lambda x: x[1], reverse=True)
    
    # Display top infosets
    for i, (infoset, total_regret) in enumerate(sorted_infosets[:num_infosets], 1):
        print(f"Infoset {i} (total regret: {total_regret:.2f}):")
        print(f"  {infoset}")
        print()
        
        # Get regrets for this infoset
        regrets = regret_table[infoset]
        
        # Sort actions by regret (descending)
        sorted_actions = sorted(regrets.items(), key=lambda x: x[1], reverse=True)
        
        print("  Cumulative Regrets:")
        for action, regret in sorted_actions:
            sign = "+" if regret >= 0 else ""
            print(f"    {action:20s}: {sign}{regret:8.2f}")
        
        print()
    
    print("="*80)


def print_table_summary():
    """
    Print overall statistics about the learned tables.
    """
    print("="*80)
    print("TABLE SUMMARY")
    print("="*80)
    print()
    
    num_infosets = len(regret_table)
    num_strategy_infosets = len(strategy_table)
    
    total_regret_entries = sum(len(actions) for actions in regret_table.values())
    total_strategy_entries = sum(len(actions) for actions in strategy_table.values())
    
    print(f"Regret Table:")
    print(f"  Information sets: {num_infosets}")
    print(f"  Total entries: {total_regret_entries}")
    print(f"  Avg actions per infoset: {total_regret_entries / num_infosets:.1f}" if num_infosets > 0 else "  Avg actions per infoset: N/A")
    print()
    
    print(f"Strategy Table:")
    print(f"  Information sets: {num_strategy_infosets}")
    print(f"  Total entries: {total_strategy_entries}")
    print(f"  Avg actions per infoset: {total_strategy_entries / num_strategy_infosets:.1f}" if num_strategy_infosets > 0 else "  Avg actions per infoset: N/A")
    print()
    
    # Find most visited infosets
    if strategy_table:
        infoset_visits = {
            infoset: sum(actions.values())
            for infoset, actions in strategy_table.items()
        }
        most_visited = max(infoset_visits.items(), key=lambda x: x[1])
        least_visited = min(infoset_visits.items(), key=lambda x: x[1])
        
        print(f"Visit Statistics:")
        print(f"  Most visited infoset: {most_visited[1]:.1f} visits")
        print(f"  Least visited infoset: {least_visited[1]:.1f} visits")
        print()
    
    # Find highest regret actions
    if regret_table:
        all_regrets = [
            (infoset, action, regret)
            for infoset, actions in regret_table.items()
            for action, regret in actions.items()
        ]
        highest_regret = max(all_regrets, key=lambda x: x[2])
        lowest_regret = min(all_regrets, key=lambda x: x[2])
        
        print(f"Regret Statistics:")
        print(f"  Highest regret: {highest_regret[2]:.2f} (action: {highest_regret[1]})")
        print(f"  Lowest regret: {lowest_regret[2]:.2f} (action: {lowest_regret[1]})")
    
    print("="*80)


print("✓ Table Inspection Utilities Defined")
print()
print("Available functions:")
print("  - print_strategy_table(num_infosets=10, sort_by='frequency')")
print("    Display learned average strategies")
print()
print("  - print_regret_table(num_infosets=10)")
print("    Display cumulative regrets for debugging")
print()
print("  - print_table_summary()")
print("    Show overall statistics about the tables")

✓ Table Inspection Utilities Defined

Available functions:
  - print_strategy_table(num_infosets=10, sort_by='frequency')
    Display learned average strategies

  - print_regret_table(num_infosets=10)
    Display cumulative regrets for debugging

  - print_table_summary()
    Show overall statistics about the tables


---
## Step 11: Inspect Learned Strategy

After training, you can inspect what the agent has learned using these utility functions:

### Available Functions:

1. **`print_table_summary()`**
   - Shows overall statistics: number of infosets, entries, visit counts
   - Good for getting a high-level overview of training progress

2. **`print_strategy_table(num_infosets=10, sort_by='frequency')`**
   - Displays the **average strategy** (Nash equilibrium approximation)
   - This is what the bot will use during play!
   - `sort_by='frequency'`: Show most visited infosets (common situations)
   - `sort_by='regret'`: Show highest regret infosets (learning hotspots)

3. **`print_regret_table(num_infosets=10)`**
   - Shows cumulative regrets for debugging
   - Positive regret = "I should play this action more"
   - Negative regret = "I'm playing this action too much"
   - Useful for understanding what the algorithm is learning

### Example Output:

```
Infoset 1 (visited 150.0 times):
  S2|H:As0,Ks1|B:Qs2,Js3,2s4|A:r

  Average Strategy:
    RAISE_100       : 65.3%
    CALL            : 30.2%
    FOLD            :  4.5%
```

This means: "On the flop with AK on QJ2, after a small raise, the learned strategy is to reraise 65% of the time, call 30%, and fold 5%"

In [16]:
# ============================================================================
# INSPECT LEARNED STRATEGY
# ============================================================================
# After training, uncomment and run these to analyze the learned policy

# 1. Overall summary
print_table_summary()

# 2. View learned strategies (average strategy = Nash approximation)
print_strategy_table(num_infosets=10, sort_by='frequency')

# 3. View cumulative regrets (for debugging)
print_regret_table(num_infosets=5)

# print("Uncomment the lines above after training to inspect the learned strategy!")

TABLE SUMMARY

Regret Table:
  Information sets: 0
  Total entries: 0
  Avg actions per infoset: N/A

Strategy Table:
  Information sets: 0
  Total entries: 0
  Avg actions per infoset: N/A

STRATEGY TABLE (Top 10 Infosets)

No strategy data yet. Train first!
REGRET TABLE (Top 5 Infosets by Total Regret)

No regret data yet. Train first!


In [17]:
# ============================================================================
# SAVE/LOAD STRATEGY TABLES
# ============================================================================
# Functions to persist learned strategies for use in the player bot

import pickle

def save_strategy(filepath='mccfr_strategy.pkl'):
    """
    Save the learned strategy and regret tables to disk.
    
    Args:
        filepath: Path to save the strategy tables
    """
    strategy_data = {
        'regret_table': dict(regret_table),
        'strategy_table': dict(strategy_table),
        'metadata': {
            'num_iterations': len(regret_table),
            'num_infosets': len(strategy_table),
        }
    }
    
    with open(filepath, 'wb') as f:
        pickle.dump(strategy_data, f)
    
    print(f"✓ Strategy saved to {filepath}")
    print(f"  Regret table: {len(strategy_data['regret_table'])} infosets")
    print(f"  Strategy table: {len(strategy_data['strategy_table'])} infosets")


def load_strategy(filepath='mccfr_strategy.pkl'):
    """
    Load a previously saved strategy.
    
    Args:
        filepath: Path to the saved strategy file
    
    Returns:
        Tuple of (regret_table, strategy_table)
    """
    with open(filepath, 'rb') as f:
        strategy_data = pickle.load(f)
    
    print(f"✓ Strategy loaded from {filepath}")
    print(f"  Regret table: {len(strategy_data['regret_table'])} infosets")
    print(f"  Strategy table: {len(strategy_data['strategy_table'])} infosets")
    
    return strategy_data['regret_table'], strategy_data['strategy_table']


print("✓ Save/Load Functions Defined")
print()
print("Usage:")
print("  save_strategy('mccfr_strategy.pkl')  # Save after training")
print("  regret_table, strategy_table = load_strategy('mccfr_strategy.pkl')  # Load later")

✓ Save/Load Functions Defined

Usage:
  save_strategy('mccfr_strategy.pkl')  # Save after training
  regret_table, strategy_table = load_strategy('mccfr_strategy.pkl')  # Load later


---
## Step 12: Save Strategy & Deploy Bot

Once you've trained for enough iterations, save the strategy and deploy the bot!

### 📁 Files Created:

1. **`player.py`** - The MCCFR player bot
   - Loads the saved strategy table
   - Queries learned policy for each decision
   - Uses average strategy (Nash approximation)

2. **`mccfr_strategy.pkl`** - Saved strategy (created when you run `save_strategy()`)
   - Contains `regret_table` and `strategy_table`
   - Can be loaded by the player bot

3. **`skeleton/`** - Required game engine files (already copied)

4. **`commands.json`** - Configuration for the game engine

### 🎮 How to Use:

```python
# 1. Train the agent (in this notebook)
utilities = train_mccfr(num_iterations=1000)

# 2. Save the learned strategy
save_strategy('mccfr_strategy.pkl')

# 3. Test against other bots (in terminal)
# cd /Users/nikhileshbelulkar/Documents/mit-poker-2026
# python3 main.py DEEP_CFR_OG/MCCFR python_skeleton_henry
```

### 🔍 How the Bot Works:

1. **Load Strategy**: `__init__()` loads `mccfr_strategy.pkl` on startup
2. **Encode State**: Converts game state to canonical infoset string (with suit isomorphism)
3. **Query Strategy**: Looks up the infoset in `strategy_table`
4. **Extract Probabilities**: Gets learned action probabilities
5. **Sample Action**: Randomly samples according to learned distribution

### 📊 Expected Performance:

- **After 100 iterations**: Random-ish play (not enough data)
- **After 1,000 iterations**: Basic strategic concepts learned
- **After 10,000 iterations**: Strong near-Nash equilibrium play
- **After 100,000+ iterations**: Very close to optimal GTO strategy

The more you train, the better it plays! 🚀

In [18]:
# ============================================================================
# SAVE STRATEGY FOR PLAYER BOT
# ============================================================================
# After training, run this to save the strategy for use in player.py

# Uncomment to save after training:
# save_strategy('mccfr_strategy.pkl')

print("After training, uncomment the line above to save your strategy!")
print("Then test your bot against other players using:")
print("  python3 main.py MCCFR_Player Other_Bot")

After training, uncomment the line above to save your strategy!
Then test your bot against other players using:
  python3 main.py MCCFR_Player Other_Bot


---
## Step 10: Run Training!

Now let's actually train the agent and see it learn!

In [19]:
# ============================================================================
# RUN TRAINING
# ============================================================================
# Train for a small number of iterations to test

ITRS = 1000 # number of iterations 

print("="*70)
print("TRAINING MCCFR AGENT")
print("="*70)
print()
print("This implements Algorithm 4: External Sampling with")
print("Stochastically-Weighted Averaging")
print()
print(f"Starting training with {ITRS} iteration(s)...")
print("(Each iteration plays 2 hands: one from P0's view, one from P1's view)")
print()

utilities = train_mccfr(num_iterations=ITRS, verbose=True)

print()
print("="*70)
print("TRAINING RESULTS")
print("="*70)
print()

# Analyze results
final_100_p0 = [u[0] for u in utilities[-100:]]
final_100_p1 = [u[1] for u in utilities[-100:]]

print(f"Final 100 iterations:")
print(f"  Player 0 avg: {np.mean(final_100_p0):.2f} chips/hand")
print(f"  Player 1 avg: {np.mean(final_100_p1):.2f} chips/hand")
print(f"  Sum: {np.mean(final_100_p0) + np.mean(final_100_p1):.2f}")
print(f"  (Note: Each traversal samples different game paths,")
print(f"   so individual utilities won't sum to zero.") 
print(f"   Over many iterations, both should converge toward ~0)")
print()

print(f"Strategy learned:")
print(f"  Information sets discovered: {len(regret_table)}")
print(f"  Total regret entries: {sum(len(actions) for actions in regret_table.values())}")
print(f"  Total strategy entries: {sum(len(actions) for actions in strategy_table.values())}")
print()

# Show example learned strategy
if len(strategy_table) > 0:
    print("Example learned strategies (first 3 information sets):")
    print()
    for i, (infoset, actions) in enumerate(list(strategy_table.items())[:3]):
        print(f"  Infoset {i+1}: {infoset[:60]}...")
        action_list = get_legal_actions_list(test_state)  # Dummy for getting actions
        # Show strategy probabilities
        total = sum(actions.values())
        if total > 0:
            for action_key, cum_strategy in actions.items():
                prob = cum_strategy / total
                if prob > 0.01:  # Only show actions with >1% probability
                    print(f"    {action_key}: {prob*100:.1f}%")
        print()

TRAINING MCCFR AGENT

This implements Algorithm 4: External Sampling with
Stochastically-Weighted Averaging

Starting training with 1000 iteration(s)...
(Each iteration plays 2 hands: one from P0's view, one from P1's view)

Iteration 100/1000
  Avg utility P0: 73.97 chips/hand
  Avg utility P1: 20.69 chips/hand
  Infosets learned: 4998
  Avg time/iter: 0.012s (83.0 iter/s)
  Elapsed: 1.2s | ETA: 10.8s
Iteration 200/1000
  Avg utility P0: 61.16 chips/hand
  Avg utility P1: 0.74 chips/hand
  Infosets learned: 10014
  Avg time/iter: 0.009s (106.3 iter/s)
  Elapsed: 2.1s | ETA: 7.5s
Iteration 300/1000
  Avg utility P0: 77.88 chips/hand
  Avg utility P1: 10.87 chips/hand
  Infosets learned: 14931
  Avg time/iter: 0.008s (132.5 iter/s)
  Elapsed: 2.9s | ETA: 5.3s
Iteration 400/1000
  Avg utility P0: 63.38 chips/hand
  Avg utility P1: -2.35 chips/hand
  Infosets learned: 19882
  Avg time/iter: 0.007s (137.5 iter/s)
  Elapsed: 3.6s | ETA: 4.4s
Iteration 500/1000
  Avg utility P0: 72.30 chips/

---
## Summary & Next Steps

### ✅ What We Implemented

This notebook implements **Algorithm 4: External Sampling with Stochastically-Weighted Averaging** from the pseudocode exactly, **WITH CRITICAL ABSTRACTIONS**:

1. **Suit Isomorphism** (`canonicalize_cards`): ⭐ A♠K♠ == A♥K♥ (only suit relationships matter) → ~75% reduction
2. **Card Ordering** (in `get_infoset`): ⭐ A♠K♠ == K♠A♠, Q♠J♠2♣ == 2♣J♠Q♠ (sort within hands/streets) → 6× fewer flop orderings
3. **Bet Size Abstraction** (in `get_infoset`): ⭐ Encodes raises as small/medium/large (r/R/B) based on pot size
4. **Action Abstraction** (`get_legal_actions_list`): ⭐ Discretizes raises to 3 sizes (min/mid/max)
5. **Information Set Abstraction** (`get_infoset`): Maps game states → canonical infoset strings
6. **Action Encoding** (`action_to_key`): Maps actions ↔ string keys
7. **Regret Matching** (`regret_matching`): Converts regrets → strategy (Line 6)
8. **Tables Initialization** (`regret_table`, `strategy_table`): Line 1
9. **External Sampling** (`external_sampling`): Lines 2-21 (core algorithm)
   - Terminal states: Line 3
   - Chance nodes: Line 4
   - Traversing player: Lines 5-15 (update regrets)
   - Opponent: Lines 16-21 (sample action, update strategy)
10. **Training Loop** (`train_mccfr`): Runs multiple iterations
11. **Strategy Extraction** (`get_average_strategy`): Extract Nash equilibrium

### 🎯 Key Insights

**⭐ Why suit isomorphism (card abstraction)?**
- **A♠K♠ and A♥K♥ are strategically identical** - only suit relationships matter!
- Without abstraction: 4× more states to learn
- With suit isomorphism: **~75% state space reduction**
- Result: **~4× faster learning** and less memory

**⭐ Why card ordering abstraction?**
- **Hand order doesn't matter**: A♠K♠ == K♠A♠ → sort hole cards
- **Flop order doesn't matter**: Q♠J♠2♣ == 2♣J♠Q♠ → sort flop cards (first 3 board cards)
- **But sequence across streets DOES matter**: Keep turn and river separate
- Result: **6× fewer flop orderings** (3! = 6 permutations → 1 canonical form)

**⭐ Why bet size abstraction?**
- **Bet sizing is part of the state space!** "Check-RaiseSmall-Call" ≠ "Check-RaiseLarge-Call"
- Action abstraction: Discretize raises to 3 sizes (min/mid/max) to make action space tractable
- History encoding: Encode bet sizes as **r**(small), **R**(medium), **B**(large) relative to pot
- Without this: Agent can't distinguish between different bet sizes → poor strategy!

**Why external sampling?**
- At opponent's nodes, we *sample* one action (not evaluate all)
- This makes traversal tractable for large games
- Still converges to Nash equilibrium!

**Why two tables?**
- `regret_table`: Updated when traversing player acts (counterfactual regrets)
- `strategy_table`: Updated when opponent acts (cumulative average strategy)
- **The average strategy converges to Nash, NOT the regret-matching strategy!**

**How it learns:**
- Each iteration:
  1. Traverse from P0's view → update P0's regrets
  2. Traverse from P1's view → update P1's regrets
- Both players learn simultaneously through self-play
- Strategy improves over time as regrets accumulate

### 📈 Next Steps

**1. Train Longer**
```python
# Train for 10,000 iterations
utilities = train_mccfr(num_iterations=10000, verbose=True)
```

**2. Test Against Baseline**
```python
# Play learned strategy vs random bot
# (Would need to integrate with engine.py)
```

**3. Add More Abstraction**
```python
# Currently implemented:
# ✅ Suit isomorphism (card abstraction)
# ✅ Action abstraction: 3 raise sizes (min/mid/max)
# ✅ Bet size encoding: Small/medium/large (r/R/B)

# Can also add:
# - Hand strength bucketing: Group similar hand strengths (e.g., by equity)
# - Board texture abstraction: Classify boards by wetness/connectedness
# - More granular bet sizing: 5+ buckets instead of 3
```

**4. Scale to Deep CFR**
- Current: Tabular MCCFR (stores every infoset)
- Deep CFR: Use neural networks as function approximators
- Handles much larger state spaces!

**5. Save/Load Strategy**
```python
import pickle
# Save
with open('mccfr_strategy.pkl', 'wb') as f:
    pickle.dump({'regret': regret_table, 'strategy': strategy_table}, f)

# Load
with open('mccfr_strategy.pkl', 'rb') as f:
    data = pickle.load(f)
    regret_table = data['regret']
    strategy_table = data['strategy']
```

### 📚 Correspondence to Pseudocode

| Pseudocode | Notebook Implementation |
|------------|------------------------|
| **Line 1**: Initialize r_I[a], s_I[a] | `regret_table`, `strategy_table` |
| **Line 2**: ExternalSampling(h, i) | `external_sampling(state, player)` |
| **Line 3**: if h ∈ Z return u_i(h) | `if isinstance(state, TerminalState)` |
| **Line 4**: if P(h)=c sample & recurse | Handled by engine (deck) |
| **Line 5**: Let I be infoset | `infoset = get_infoset(state, player)` |
| **Line 6**: σ(I) ← RegretMatching | `strategy = regret_matching(...)` |
| **Line 7**: if P(I)=i | `if active_player == traversing_player` |
| **Line 8**: u array, u_σ←0 | `action_values = {}`, `node_value = 0.0` |
| **Line 10**: u[a]←ExternalSampling | `action_values[a] = external_sampling(...)` |
| **Line 11**: u_σ += σ·u[a] | `node_value += strategy[a] * action_values[a]` |
| **Line 13**: r̃(I,a)←u[a]-u_σ | `regret = action_values[a] - node_value` |
| **Line 14**: r_I[a] += r̃(I,a) | `regret_table[infoset][a] += regret` |
| **Line 15**: return u_σ | `return node_value` |
| **Line 17**: Sample a' from σ(I) | `sampled_action = random.choices(...)` |
| **Line 18**: u←ExternalSampling | `value = external_sampling(...)` |
| **Line 20**: s_I[a] += σ(I,a) | `strategy_table[infoset][a] += strategy[a]` |
| **Line 21**: return u | `return value` |

---

**Congratulations! You've implemented tabular MCCFR from scratch! 🎉**

---
## Bonus: Inspect Learned Strategy

Let's see what strategies the agent learned for specific situations!

In [46]:
# ============================================================================
# INSPECT LEARNED STRATEGY
# ============================================================================
# Analyze what the agent learned

def inspect_strategy_sample():
    """Show some example learned strategies."""
    print("="*70)
    print("LEARNED STRATEGY EXAMPLES")
    print("="*70)
    print()
    
    # Sort information sets by how often they were visited
    infoset_visit_counts = {
        infoset: sum(strategy_table[infoset].values())
        for infoset in strategy_table.keys()
    }
    
    # Get top 10 most visited infosets
    top_infosets = sorted(infoset_visit_counts.items(), 
                         key=lambda x: x[1], 
                         reverse=True)[:10]
    
    print(f"Top 10 most frequently encountered situations:")
    print()
    
    for i, (infoset, visit_count) in enumerate(top_infosets, 1):
        print(f"{i}. {infoset[:80]}")
        print(f"   Visited: {visit_count:.0f} times")
        
        # Get strategy for this infoset
        actions = strategy_table[infoset]
        total = sum(actions.values())
        
        if total > 0:
            print("   Strategy:")
            # Sort actions by probability
            sorted_actions = sorted(
                actions.items(), 
                key=lambda x: x[1]/total, 
                reverse=True
            )
            
            for action_key, cum_strategy in sorted_actions[:5]:  # Top 5 actions
                prob = cum_strategy / total
                if prob > 0.001:  # Only show >0.1%
                    print(f"     {action_key:20s}: {prob*100:5.1f}%")
        
        print()
    
    print("="*70)
    
    # Statistics
    print("\nOverall Statistics:")
    print(f"  Total information sets: {len(strategy_table)}")
    print(f"  Avg actions per infoset: {np.mean([len(a) for a in strategy_table.values()]):.1f}")
    print(f"  Total strategy updates: {sum(infoset_visit_counts.values()):.0f}")


# Run inspection
inspect_strategy_sample()

LEARNED STRATEGY EXAMPLES

Top 10 most frequently encountered situations:

1. S6|H:2s3,5s3|B:7s3,8s1,9s1,9s2,Ks0,Ts0|A:rBrDXDXrrrrrrr
   Visited: 34 times
   Strategy:
     FOLD                :  34.6%
     CALL                :  24.3%
     CHECK               :  10.3%
     RAISE_4             :   4.7%
     RAISE_190           :   4.7%

2. S6|H:4s1,6s0|B:2s3,5s0,6s2,7s1,Js1,Ts0|A:rBrDXDXrrrrrrrr
   Visited: 28 times
   Strategy:
     FOLD                :  37.3%
     CALL                :  29.4%
     CHECK               :   7.9%
     RAISE_94            :   3.8%
     RAISE_4             :   2.9%

3. S6|H:5s1,8s0|B:2s0,3s1,4s2,5s0,8s1,8s2|A:rrBrDXDXrrrrrr
   Visited: 28 times
   Strategy:
     FOLD                :  36.2%
     CALL                :  25.8%
     CHECK               :  10.4%
     RAISE_95            :   8.3%
     RAISE_4             :   5.0%

4. S6|H:2s0,7s2|B:4s1,8s1,8s2,Ks0,Qs1,Ts0|A:rBrDXDXrrrrrr
   Visited: 27 times
   Strategy:
     FOLD                :  36.2%
     C

In [47]:
def train_mccfr_with_checkpoints(
    num_iterations: int,
    checkpoint_interval: int = 100_000,
    resume_from: str = None,
    verbose: bool = True
):
    """Train MCCFR with checkpoint saving every N iterations."""
    start_iteration = 0
    utilities = []
    
    # Resume from checkpoint if specified
    if resume_from:
        print(f"Resuming from: {resume_from}")
        try:
            with open(resume_from, 'rb') as f:
                data = pickle.load(f)
            global regret_table, strategy_table
            regret_table = defaultdict(float, data['regret_table'])
            strategy_table = defaultdict(lambda: defaultdict(float),
                {k: defaultdict(float, v) for k, v in data['strategy_table'].items()})
            start_iteration = data['iteration']
            utilities = data.get('utilities', [])
            print(f"✓ Resumed from iteration {start_iteration:,}")
        except Exception as e:
            print(f"✗ Failed to resume: {e}")
    
    start_time = time.time()
    iteration_times = []
    
    print("="*80)
    print("🌙 OVERNIGHT TRAINING WITH CHECKPOINTS")
    print("="*80)
    print(f"Total iterations: {num_iterations:,}")
    print(f"Checkpoint every: {checkpoint_interval:,}")
    print(f"Estimated time: {num_iterations * 0.015 / 3600:.1f} hours")
    print("="*80 + "\n")
    
    for iteration in range(start_iteration, num_iterations):
        iter_start = time.time()
        
        initial_state = create_initial_state()
        utility_p0 = external_sampling(initial_state, 0)
        utility_p1 = external_sampling(initial_state, 1)
        utilities.append((utility_p0, utility_p1))
        
        iteration_times.append(time.time() - iter_start)
        
        # Progress
        if verbose and (iteration + 1) % 100 == 0:
            avg_p0 = np.mean([u[0] for u in utilities[-100:]])
            avg_p1 = np.mean([u[1] for u in utilities[-100:]])
            avg_time = np.mean(iteration_times[-100:])
            elapsed = time.time() - start_time
            eta = avg_time * (num_iterations - iteration - 1)
            
            print(f"[{iteration + 1:,}/{num_iterations:,}] "\
                  f"P0:{avg_p0:+.1f} P1:{avg_p1:+.1f} | "\
                  f"States:{len(strategy_table):,} | "\
                  f"{1/avg_time:.0f}it/s | "\
                  f"⏱{elapsed/60:.0f}m | ETA:{eta/3600:.1f}h")
        
        # Save checkpoint (using same approach as save_strategy)
        if (iteration + 1) % checkpoint_interval == 0:
            checkpoint_data = {
                'regret_table': dict(regret_table),
                'strategy_table': dict(strategy_table),
                'iteration': iteration + 1,
                'utilities': utilities,
                'timestamp': time.time()
            }
            
            # Save numbered checkpoint + latest
            checkpoint_file = f"mccfr_checkpoint_{iteration+1}.pkl"
            latest_file = "mccfr_strategy.pkl"
            
            with open(checkpoint_file, 'wb') as f:
                pickle.dump(checkpoint_data, f)
            with open(latest_file, 'wb') as f:
                pickle.dump(checkpoint_data, f)
            
            size_mb = Path(checkpoint_file).stat().st_size / 1024**2
            print(f"\n💾 CHECKPOINT {iteration+1:,} | {size_mb:.0f}MB saved\n")
    
    # Final save
    total_time = time.time() - start_time
    final_data = {
        'regret_table': dict(regret_table),
        'strategy_table': dict(strategy_table),
        'iteration': num_iterations,
        'utilities': utilities,
        'total_time': total_time
    }
    
    with open(f"mccfr_final_{num_iterations}.pkl", 'wb') as f:
        pickle.dump(final_data, f)
    with open("mccfr_strategy.pkl", 'wb') as f:
        pickle.dump(final_data, f)
    
    print(f"\n{'='*80}")
    print("🎉 TRAINING COMPLETE!")
    print(f"{'='*80}")
    print(f"Time: {total_time/3600:.2f}h | States: {len(strategy_table):,}")
    print(f"{'='*80}\n")
    
    return utilities

print("✓ Checkpoint training ready")

✓ Checkpoint training ready


---
## 🚀 START 10M ITERATION TRAINING

**Run this cell to start overnight training!**

- Saves every 100k iterations to `mccfr_strategy.pkl` (bot uses this)
- Keeps numbered backups: `mccfr_checkpoint_100000.pkl`, etc.
- Can resume if interrupted
- Estimated time: ~12-42 hours

In [ ]:
# 🌙 10 MILLION ITERATION TRAINING
# Run this cell and let it train overnight!

TOTAL_ITERATIONS = 2000
CHECKPOINT_EVERY = 1000

# To resume: RESUME_FROM = 'mccfr_checkpoint_500000.pkl'
RESUME_FROM = None

# START TRAINING
utilities = train_mccfr_with_checkpoints(
    num_iterations=TOTAL_ITERATIONS,
    checkpoint_interval=CHECKPOINT_EVERY,
    resume_from=RESUME_FROM,
    verbose=True
)

# Final statistics
print("\n" + "="*80)
print("FINAL STATISTICS")
print("="*80)

avg_p0 = np.mean([u[0] for u in utilities[-1000:]])
avg_p1 = np.mean([u[1] for u in utilities[-1000:]])
visit_counts = {k: sum(v.values()) for k, v in strategy_table.items()}
avg_visits = np.mean(list(visit_counts.values()))
max_visits = max(visit_counts.values())

print(f"Last 1000 hands: P0:{avg_p0:+.2f} P1:{avg_p1:+.2f} Sum:{avg_p0+avg_p1:.2f}")
print(f"States learned: {len(strategy_table):,}")
print(f"Avg visits/state: {avg_visits:.1f}")
print(f"Max visits: {max_visits:.0f}")

if avg_visits > 20:
    print("\n✅ EXCELLENT! Strong strategy learned (avg visits > 20)")
elif avg_visits > 10:
    print("\n✓ GOOD! Decent strategy (avg visits > 10)")
else:
    print("\n⚠️ Still sparse - consider more training")

print("\n" + "="*80)
print("Files saved:")
print(f"  Latest: mccfr_strategy.pkl (your bot uses this)")
print(f"  Final: mccfr_final_{TOTAL_ITERATIONS}.pkl")
print(f"  Checkpoints: mccfr_checkpoint_*.pkl")
print("="*80)

🌙 OVERNIGHT TRAINING WITH CHECKPOINTS
Total iterations: 10,000,000,000,000
Checkpoint every: 50,000
Estimated time: 41666666.7 hours

[100/10,000,000,000,000] P0:+37.9 P1:-48.3 | States:423,459 | 42it/s | ⏱0m | ETA:66428045.4h
[200/10,000,000,000,000] P0:+31.0 P1:-30.9 | States:440,567 | 40it/s | ⏱0m | ETA:69613860.7h
[300/10,000,000,000,000] P0:+28.6 P1:-32.5 | States:455,563 | 49it/s | ⏱0m | ETA:56761086.0h
[400/10,000,000,000,000] P0:+32.3 P1:-42.6 | States:471,040 | 42it/s | ⏱0m | ETA:65370976.9h
[500/10,000,000,000,000] P0:+32.2 P1:-33.6 | States:487,198 | 46it/s | ⏱0m | ETA:59782948.8h
[600/10,000,000,000,000] P0:+44.3 P1:-30.2 | States:503,740 | 44it/s | ⏱0m | ETA:63174267.6h
[700/10,000,000,000,000] P0:+30.9 P1:-37.4 | States:520,162 | 62it/s | ⏱0m | ETA:44655508.5h
[800/10,000,000,000,000] P0:+44.3 P1:-45.7 | States:536,840 | 50it/s | ⏱0m | ETA:55465605.5h
[900/10,000,000,000,000] P0:+41.6 P1:-36.7 | States:551,287 | 73it/s | ⏱0m | ETA:38124779.9h
[1,000/10,000,000,000,000] P0

: 